In [1]:
import os
import math
import copy
import csv
import time

import numpy as np
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
timesteps = 1000


def cosine_beta_schedule(
    timesteps,
    s=0.008
):
    steps = timesteps + 1

    x = torch.linspace(
        0,
        timesteps,
        steps,
        dtype=torch.float64
    )

    alpha_bar = torch.cos(
        (
            (x / timesteps + s)
            / (1 + s)
        )
        * math.pi
        * 0.5
    ) ** 2

    alpha_bar = (
        alpha_bar
        / alpha_bar[0]
    )

    betas = (
        1.0
        - (
            alpha_bar[1:]
            / alpha_bar[:-1]
        )
    )

    return torch.clamp(
        betas,
        min=1e-8,
        max=0.999
    ).float()


def rescale_zero_terminal_snr(
    betas
):
    """
    Algorithm 1 from:
    'Common Diffusion Noise Schedules
    and Sample Steps are Flawed'

    Rescales beta schedule so that
    alpha_bar_T = 0 exactly.
    """

    alphas = (
        1.0 - betas
    )

    alphas_cumprod = torch.cumprod(
        alphas,
        dim=0
    )

    alpha_bar_sqrt = torch.sqrt(
        alphas_cumprod
    )

    alpha_bar_sqrt_0 = (
        alpha_bar_sqrt[0].clone()
    )

    alpha_bar_sqrt_T = (
        alpha_bar_sqrt[-1].clone()
    )

    # Shift terminal value to zero
    alpha_bar_sqrt = (
        alpha_bar_sqrt
        - alpha_bar_sqrt_T
    )

    # Preserve initial value
    alpha_bar_sqrt = (
        alpha_bar_sqrt
        * alpha_bar_sqrt_0
        / (
            alpha_bar_sqrt_0
            - alpha_bar_sqrt_T
        )
    )

    alpha_bar = (
        alpha_bar_sqrt ** 2
    )

    new_alphas = (
        alpha_bar[1:]
        / alpha_bar[:-1]
    )

    new_alphas = torch.cat(
        [
            alpha_bar[0:1],
            new_alphas
        ]
    )

    new_betas = (
        1.0 - new_alphas
    )

    return new_betas.float()


# Original cosine schedule
betas = cosine_beta_schedule(
    timesteps
)

# V5: force terminal SNR to exactly zero
betas = rescale_zero_terminal_snr(
    betas
)


alphas = (
    1.0 - betas
)

alphas_cumprod = torch.cumprod(
    alphas,
    dim=0
)

alphas_cumprod_prev = F.pad(
    alphas_cumprod[:-1],
    (1, 0),
    value=1.0
)


sqrt_alphas_cumprod = torch.sqrt(
    alphas_cumprod
)

sqrt_one_minus_alphas_cumprod = (
    torch.sqrt(
        1.0
        - alphas_cumprod
    )
)


posterior_variance = (
    betas
    * (
        1.0
        - alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_variance = torch.clamp(
    posterior_variance,
    min=1e-20
)


posterior_mean_coef1 = (
    betas
    * torch.sqrt(
        alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)


posterior_mean_coef2 = (
    (
        1.0
        - alphas_cumprod_prev
    )
    * torch.sqrt(
        alphas
    )
    / (
        1.0
        - alphas_cumprod
    )
)


snr = (
    alphas_cumprod
    / torch.clamp(
        1.0
        - alphas_cumprod,
        min=1e-12
    )
)


print(
    "Beta range:",
    betas.min().item(),
    betas.max().item()
)

print(
    "Initial alpha_cumprod:",
    alphas_cumprod[0].item()
)

print(
    "Final alpha_cumprod:",
    alphas_cumprod[-1].item()
)

print(
    "Final SNR:",
    snr[-1].item()
)


assert (
    alphas_cumprod[-1].item()
    == 0.0
)

print(
    "Zero-terminal-SNR check passed."
)

Beta range: 4.124641418457031e-05 1.0
Initial alpha_cumprod: 0.9999587535858154
Final alpha_cumprod: 0.0
Final SNR: 0.0
Zero-terminal-SNR check passed.


In [3]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(
        self,
        dim
    ):
        super().__init__()

        self.dim = dim

    def forward(self, t):

        device = t.device

        half_dim = (
            self.dim // 2
        )

        embedding_scale = (
            math.log(10000)
            / (half_dim - 1)
        )

        embeddings = torch.exp(
            torch.arange(
                half_dim,
                device=device
            )
            * -embedding_scale
        )

        embeddings = (
            t[:, None].float()
            * embeddings[None, :]
        )

        embeddings = torch.cat(
            (
                embeddings.sin(),
                embeddings.cos()
            ),
            dim=1
        )

        return embeddings

In [4]:
class ResBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=in_channels
        )

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                time_dim,
                out_channels * 2
            )
        )

        # V5:
        # Start scale = 0 and shift = 0
        nn.init.zeros_(
            self.time_mlp[-1].weight
        )

        nn.init.zeros_(
            self.time_mlp[-1].bias
        )


        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        # Start residual branch near zero
        nn.init.zeros_(
            self.conv2.weight
        )

        nn.init.zeros_(
            self.conv2.bias
        )


        if in_channels != out_channels:
            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()


    def forward(
        self,
        x,
        t
    ):
        residual = self.residual(
            x
        )

        h = self.norm1(
            x
        )

        h = F.silu(
            h
        )

        h = self.conv1(
            h
        )

        time_emb = self.time_mlp(
            t
        )

        scale, shift = (
            time_emb.chunk(
                2,
                dim=1
            )
        )

        scale = scale[
            :,
            :,
            None,
            None,
            None
        ]

        shift = shift[
            :,
            :,
            None,
            None,
            None
        ]

        h = self.norm2(
            h
        )

        h = (
            h
            * (
                1.0 + scale
            )
            + shift
        )

        h = F.silu(
            h
        )

        h = self.dropout(
            h
        )

        h = self.conv2(
            h
        )

        return (
            h + residual
        )

In [5]:
class DownBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.resblock1 = ResBlock3D(
            in_channels,
            out_channels,
            time_dim
        )

        self.resblock2 = ResBlock3D(
            out_channels,
            out_channels,
            time_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(
        self,
        x,
        t
    ):
        x = self.resblock1(
            x,
            t
        )

        x = self.resblock2(
            x,
            t
        )

        skip = x

        x = self.downsample(
            x
        )

        return skip, x


class UpBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock1 = ResBlock3D(
            out_channels
            + skip_channels,
            out_channels,
            time_dim
        )

        self.resblock2 = ResBlock3D(
            out_channels,
            out_channels,
            time_dim
        )

    def forward(
        self,
        x,
        skip,
        t
    ):
        x = self.upsample(
            x
        )

        if x.shape[2:] != skip.shape[2:]:
            raise ValueError(
                f"Upsample shape {x.shape} "
                f"does not match skip {skip.shape}"
            )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.resblock1(
            x,
            t
        )

        x = self.resblock2(
            x,
            t
        )

        return x

In [6]:
class AttentionBlock3D(nn.Module):
    def __init__(
        self,
        channels,
        num_heads=4
    ):
        super().__init__()

        if channels % num_heads != 0:
            raise ValueError(
                "channels must be divisible "
                "by num_heads"
            )

        self.norm = nn.GroupNorm(
            num_groups=8,
            num_channels=channels
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        b, c, d, h, w = x.shape

        residual = x

        x = self.norm(
            x
        )

        # [B,C,D,H,W]
        # ->
        # [B,D*H*W,C]
        x = (
            x.permute(
                0,
                2,
                3,
                4,
                1
            )
            .reshape(
                b,
                d * h * w,
                c
            )
        )

        x, _ = self.attention(
            x,
            x,
            x,
            need_weights=False
        )

        # Restore 3D shape
        x = (
            x.reshape(
                b,
                d,
                h,
                w,
                c
            )
            .permute(
                0,
                4,
                1,
                2,
                3
            )
            .contiguous()
        )

        return x + residual


class UNet3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=16,
        time_dim=256
    ):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(
                time_dim
            ),
            nn.Linear(
                time_dim,
                time_dim
            ),
            nn.SiLU(),
            nn.Linear(
                time_dim,
                time_dim
            )
        )

        # 208 x 224 x 160
        self.input_conv = nn.Conv3d(
            in_channels,
            base_channels,
            kernel_size=3,
            padding=1
        )

        # 208x224x160 -> 104x112x80
        self.down1 = DownBlock3D(
            base_channels,
            base_channels * 2,
            time_dim
        )

        # 104x112x80 -> 52x56x40
        self.down2 = DownBlock3D(
            base_channels * 2,
            base_channels * 4,
            time_dim
        )

        # 52x56x40 -> 26x28x20
        self.down3 = DownBlock3D(
            base_channels * 4,
            base_channels * 8,
            time_dim
        )

        # 26x28x20 -> 13x14x10
        self.down4 = DownBlock3D(
            base_channels * 8,
            base_channels * 16,
            time_dim
        )

        # Bottleneck: 13 x 14 x 10
        self.mid1 = ResBlock3D(
            base_channels * 16,
            base_channels * 16,
            time_dim
        )

        self.mid_attention = AttentionBlock3D(
            base_channels * 16,
            num_heads=4
        )

        self.mid2 = ResBlock3D(
            base_channels * 16,
            base_channels * 16,
            time_dim
        )

        # 13x14x10 -> 26x28x20
        self.up4 = UpBlock3D(
            in_channels=base_channels * 16,
            skip_channels=base_channels * 16,
            out_channels=base_channels * 8,
            time_dim=time_dim
        )

        # 26x28x20 -> 52x56x40
        self.up3 = UpBlock3D(
            in_channels=base_channels * 8,
            skip_channels=base_channels * 8,
            out_channels=base_channels * 4,
            time_dim=time_dim
        )

        # 52x56x40 -> 104x112x80
        self.up2 = UpBlock3D(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 4,
            out_channels=base_channels * 2,
            time_dim=time_dim
        )

        # 104x112x80 -> 208x224x160
        self.up1 = UpBlock3D(
            in_channels=base_channels * 2,
            skip_channels=base_channels * 2,
            out_channels=base_channels,
            time_dim=time_dim
        )

        self.output_norm = nn.GroupNorm(
            num_groups=8,
            num_channels=base_channels
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        # Start final noise-prediction layer near zero
        nn.init.zeros_(
            self.output_conv.weight
        )

        nn.init.zeros_(
            self.output_conv.bias
        )

    def forward(
        self,
        x,
        t
    ):
        t = self.time_embedding(
            t
        )

        x = self.input_conv(
            x
        )

        skip1, x = self.down1(
            x,
            t
        )

        skip2, x = self.down2(
            x,
            t
        )

        skip3, x = self.down3(
            x,
            t
        )

        skip4, x = self.down4(
            x,
            t
        )

        x = self.mid1(
            x,
            t
        )

        x = self.mid_attention(
            x
        )

        x = self.mid2(
            x,
            t
        )

        x = self.up4(
            x,
            skip4,
            t
        )

        x = self.up3(
            x,
            skip3,
            t
        )

        x = self.up2(
            x,
            skip2,
            t
        )

        x = self.up1(
            x,
            skip1,
            t
        )

        x = self.output_norm(
            x
        )

        x = F.silu(
            x
        )

        x = self.output_conv(
            x
        )

        return x

In [7]:
class EMA:
    def __init__(
        self,
        model,
        decay=0.9999
    ):
        self.decay = decay

        self.ema_model = copy.deepcopy(model)
        self.ema_model.eval()

        for parameter in self.ema_model.parameters():
            parameter.requires_grad = False


    @torch.no_grad()
    def update(
        self,
        model
    ):
        ema_parameters = dict(
            self.ema_model.named_parameters()
        )

        model_parameters = dict(
            model.named_parameters()
        )

        for name, parameter in model_parameters.items():
            ema_parameters[name].mul_(self.decay).add_(
                parameter,
                alpha=(1.0 - self.decay)
            )

        ema_buffers = dict(
            self.ema_model.named_buffers()
        )

        model_buffers = dict(
            model.named_buffers()
        )

        for name, buffer in model_buffers.items():
            ema_buffers[name].copy_(buffer)

In [8]:
def load_checkpoint(
    model,
    ema,
    path,
    device
):
    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    ema.ema_model.load_state_dict(
        checkpoint["ema_state_dict"]
    )

    return checkpoint["epoch"]

In [9]:
@torch.no_grad()
def sample_ddpm(
    model,
    shape,
    device
):
    model.eval()

    # Inference starts from pure Gaussian noise
    # at the actual final training timestep
    x = torch.randn(
        shape,
        device=device
    )


    sqrt_alpha_bar = (
        sqrt_alphas_cumprod
        .to(device)
    )

    sqrt_one_minus_alpha_bar = (
        sqrt_one_minus_alphas_cumprod
        .to(device)
    )

    coef1 = (
        posterior_mean_coef1
        .to(device)
    )

    coef2 = (
        posterior_mean_coef2
        .to(device)
    )

    posterior_var = (
        posterior_variance
        .to(device)
    )


    for t in reversed(
        range(timesteps)
    ):
        t_batch = torch.full(
            (
                shape[0],
            ),
            t,
            device=device,
            dtype=torch.long
        )


        # Model predicts velocity v
        v_pred = model(
            x,
            t_batch
        )


        # Recover clean x0 directly
        # from v-prediction.
        #
        # x0 =
        # sqrt(alpha_bar) * xt
        # -
        # sqrt(1-alpha_bar) * v
        x0_pred = (
            sqrt_alpha_bar[t]
            * x
            -
            sqrt_one_minus_alpha_bar[t]
            * v_pred
        )


        x0_pred = torch.clamp(
            x0_pred,
            -1.0,
            1.0
        )


        model_mean = (
            coef1[t]
            * x0_pred
            +
            coef2[t]
            * x
        )


        if t > 0:

            noise = torch.randn_like(
                x
            )

            x = (
                model_mean
                +
                torch.sqrt(
                    posterior_var[t]
                )
                * noise
            )

        else:

            x = model_mean


    return torch.clamp(
        x,
        -1.0,
        1.0
    )

In [10]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=256
).to(device)


ema = EMA(
    model,
    decay=0.9999
)


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print("Prediction target: v-prediction")
print("Foreground auxiliary lambda: 0.5")
print("Zero-terminal-SNR: enabled")

Device: cuda
Total parameters: 28,706,065
Trainable parameters: 28,706,065
Prediction target: v-prediction
Foreground auxiliary lambda: 0.5
Zero-terminal-SNR: enabled


In [11]:
CKPT_PATH = (
    "ddpm_v5_checkpoints/"
    "ddpm_v5_epoch_050.pt"
)

loaded_epoch = load_checkpoint(
    model=model,
    ema=ema,
    path=CKPT_PATH,
    device=device
)

print("Loaded V5 epoch:", loaded_epoch)

Loaded V5 epoch: 50


In [12]:
NUM_SAMPLES = 200
BASE_SEED = 10000

OUTPUT_DIR = "evaluation_200/ddpm_v5"
METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "metadata_ddpm_v5.csv"
)

SAMPLE_SHAPE = (
    1,
    1,
    208,
    224,
    160
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Number of samples:", NUM_SAMPLES)
print("Output directory:", OUTPUT_DIR)
print("Metadata path:", METADATA_PATH)
print(
    "Seed range:",
    BASE_SEED,
    "to",
    BASE_SEED + NUM_SAMPLES - 1
)

Number of samples: 1
Output directory: evaluation_200/ddpm_v5
Metadata path: evaluation_200/ddpm_v5/metadata_ddpm_v5.csv
Seed range: 10000 to 10000


In [13]:
metadata_exists = os.path.exists(
    METADATA_PATH
)

if not metadata_exists:
    with open(
        METADATA_PATH,
        "w",
        newline=""
    ) as f:
        writer = csv.writer(f)
        writer.writerow([
            "sample_id",
            "filename",
            "seed",
            "shape_x",
            "shape_y",
            "shape_z",
            "min",
            "max",
            "mean",
            "std",
            "generation_seconds"
        ])


total_start = time.perf_counter()
generated_this_run = 0

for i in range(NUM_SAMPLES):

    sample_id = f"{i:04d}"
    seed = BASE_SEED + i
    filename = f"ddpm_v5_{sample_id}.nii.gz"
    output_path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(output_path):
        print(
            f"[{i + 1:03d}/{NUM_SAMPLES}] "
            f"{filename} already exists -> skipped"
        )
        continue

    print()
    print(
        f"[{i + 1:03d}/{NUM_SAMPLES}] "
        f"Generating {filename}"
    )
    print(f"Seed: {seed}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    sample_start = time.perf_counter()

    generated = sample_ddpm(
        model=ema.ema_model,
        shape=SAMPLE_SHAPE,
        device=device
    )

    if device.type == "cuda":
        torch.cuda.synchronize()

    sample_seconds = (
        time.perf_counter()
        - sample_start
    )

    volume = (
        generated[0, 0]
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    # Convert from [-1,1] to [0,1]
    volume = (volume + 1.0) / 2.0
    volume = np.clip(
        volume,
        0.0,
        1.0
    ).astype(np.float32)

    expected_shape = (
        208,
        224,
        160
    )

    if volume.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected shape: {volume.shape}"
        )

    if not np.all(np.isfinite(volume)):
        raise RuntimeError(
            f"NaN or Inf found in sample {sample_id}"
        )

    volume_min = float(volume.min())
    volume_max = float(volume.max())
    volume_mean = float(volume.mean())
    volume_std = float(volume.std())

    affine = np.eye(4, dtype=np.float32)

    nifti_image = nib.Nifti1Image(
        volume,
        affine
    )
    nifti_image.set_data_dtype(np.float32)

    nib.save(
        nifti_image,
        output_path
    )

    with open(
        METADATA_PATH,
        "a",
        newline=""
    ) as f:
        writer = csv.writer(f)
        writer.writerow([
            sample_id,
            filename,
            seed,
            volume.shape[0],
            volume.shape[1],
            volume.shape[2],
            volume_min,
            volume_max,
            volume_mean,
            volume_std,
            sample_seconds
        ])

    generated_this_run += 1

    completed_files = len([
        f for f in os.listdir(OUTPUT_DIR)
        if f.startswith("ddpm_v5_")
        and f.endswith(".nii.gz")
    ])

    remaining = NUM_SAMPLES - completed_files
    estimated_remaining_hours = (
        remaining * sample_seconds / 3600.0
    )

    print(f"Saved: {output_path}")
    print("Shape:", volume.shape)
    print("Range:", volume_min, volume_max)
    print("Mean:", volume_mean)
    print("Std:", volume_std)
    print(
        f"Generation time: "
        f"{sample_seconds / 60:.2f} min"
    )
    print(
        f"Completed: "
        f"{completed_files}/{NUM_SAMPLES}"
    )
    print(
        f"Estimated remaining time: "
        f"{estimated_remaining_hours:.2f} h"
    )

    del generated
    del volume
    del nifti_image

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


total_seconds = (
    time.perf_counter()
    - total_start
)

print()
print("========================================")
print("DDPM V5 generation finished")
print("========================================")
print(
    "Generated during this run:",
    generated_this_run
)
print(
    f"Runtime this session: "
    f"{total_seconds / 3600:.2f} h"
)
print("Output directory:", OUTPUT_DIR)
print("Metadata:", METADATA_PATH)


[001/1] Generating ddpm_v5_0000.nii.gz
Seed: 10000
Saved: evaluation_200/ddpm_v5/ddpm_v5_0000.nii.gz
Shape: (208, 224, 160)
Range: 0.0 1.0
Mean: 0.10535803437232971
Std: 0.1838265359401703
Generation time: 9.85 min
Completed: 1/1
Estimated remaining time: 0.00 h

DDPM V5 generation finished
Generated during this run: 1
Runtime this session: 0.16 h
Output directory: evaluation_200/ddpm_v5
Metadata: evaluation_200/ddpm_v5/metadata_ddpm_v5.csv
